**ATTENTION AND TRANSFORMERS**

**TRANSFORMER ARCHITECTURE FROM NUMPY**

In [84]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [85]:
x = np.random.randn(5,4)
x

array([[-0.2043899 ,  0.68873344,  0.10442207, -0.67174051],
       [ 0.33497862,  0.21376939,  2.69201814, -0.84693855],
       [ 0.85012931,  0.29976682,  0.88607554,  0.09629256],
       [-0.02334654, -1.33519976,  0.13068198, -1.32551405],
       [ 1.74008238,  1.76985395,  0.65084499,  0.49854367]])

In [86]:
wq = np.random.rand(4,4)
wk = np.random.rand(4,4)
wv = np.random.rand(4,4)

In [87]:
Q = np.dot(x,wq)
K = np.dot(x,wk)
V = np.dot(x,wv)

In [88]:
raw_score = (Q @ K.T)/np.sqrt(4)

In [89]:
raw_score.shape

(5, 5)

In [90]:
softmax = np.exp(raw_score) / np.sum(np.exp(raw_score),axis=1,keepdims=True)

In [91]:
output = np.dot(softmax,V)
output.shape

(5, 4)

In [92]:
print(np.sum(softmax, axis=1))

[1. 1. 1. 1. 1.]


**ATTENTION AND TRANSFORMER APPLIED TO JPMC AND S&P500 DATA**

**PREPARING THE DATA**

In [93]:
jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')

/tmp/ipykernel_1481/4128732890.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  jpmc = yf.download(['JPM'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1481/4128732890.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download(['^GSPC'],start = '2010-01-1',end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


**EXPLORATORY DATA ANALYSIS**

In [94]:
jpmc.isnull().sum().sum()

np.int64(0)

In [95]:
sp500.isnull().sum().sum()

np.int64(0)

In [96]:
jpmc.duplicated().sum()

np.int64(0)

In [97]:
sp500.duplicated().sum()

np.int64(0)

In [98]:
jpmc.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (Close, JPM)   3774 non-null   float64
 1   (High, JPM)    3774 non-null   float64
 2   (Low, JPM)     3774 non-null   float64
 3   (Open, JPM)    3774 non-null   float64
 4   (Volume, JPM)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


In [99]:
sp500.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3774 entries, 2010-01-04 to 2024-12-31
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (Close, ^GSPC)   3774 non-null   float64
 1   (High, ^GSPC)    3774 non-null   float64
 2   (Low, ^GSPC)     3774 non-null   float64
 3   (Open, ^GSPC)    3774 non-null   float64
 4   (Volume, ^GSPC)  3774 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 176.9 KB


**FEATURE ENGINEERING**

In [100]:
#Daily returns
jpmc['returns'] = jpmc['Close']['JPM'].pct_change(fill_method=None)

In [101]:
#volume ratio
volume = jpmc['Volume']['JPM']
jpmc['volume ratio'] = (volume/ volume.rolling(window=20).mean()).shift(1)

In [102]:
#20-day rolling return
jpmc['20-day rolling return'] = jpmc['returns'].rolling(window=20).mean().shift(1)

In [103]:
delta = jpmc['Close']['JPM'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
jpmc['RSI'] = RSI.shift(1)

In [104]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [105]:
sp500_return_dataframe = pd.DataFrame({
    'Date': sp500.index,
    'SP500 returns': sp500['SP500 returns']
})
sp500_return_dataframe.set_index('Date', inplace=True)

In [106]:
jpmc_clean = pd.DataFrame({
    "returns": jpmc['returns'],
    "Volume Ratio": jpmc['volume ratio'],
    "Rolling Returns": jpmc['20-day rolling return'],
    "RSI": jpmc['RSI']
})

In [107]:
jpmc_clean = jpmc_clean.join(sp500_return_dataframe,how='inner')

In [108]:
jpmc_clean['target'] = (jpmc_clean['returns'] > 0).astype(int).shift(-1)

In [109]:
jpmc_clean.dropna(inplace=True)
jpmc_clean

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns,target
Date,,,,,,
2010-02-03,-0.006411,0.853654,-0.002507,36.612114,-0.005474,0.0
2010-02-04,-0.048151,0.696517,-0.003796,31.107016,-0.031141,0.0
2010-02-05,-0.001304,1.037733,-0.006478,23.539320,0.002897,0.0
2010-02-08,-0.015666,1.327887,-0.007534,25.589928,-0.008863,1.0
2010-02-09,0.018302,1.007097,-0.008194,25.133829,0.013040,1.0
...,...,...,...,...,...,...
2024-12-23,0.003325,3.523394,-0.001417,36.638927,0.007287,1.0
2024-12-24,0.016444,0.934839,-0.002024,39.867659,0.011043,1.0
2024-12-26,0.003425,0.419782,-0.001552,48.407856,-0.000406,0.0


In [110]:
jpmc_clean.shape

(3752, 6)

In [111]:
jpmc_clean.columns

Index(['returns', 'Volume Ratio', 'Rolling Returns', 'RSI', 'SP500 returns',
       'target'],
      dtype='object')